# Resultados XAI avanzados para Kepler y TESS

Este notebook organiza los resultados principales de interpretabilidad de los mejores modelos. No reentrena los modelos: usa los checkpoints existentes, las tablas ya generadas en `resultados_xai` y las salidas adicionales preparadas en `notebook/resultados`.

Incluye:

- mapas de importancia y fidelidad por borrado;
- conteo de focos de atencion por muestra y por matriz de confusion;
- comparacion MaxLike vs MC Dropout;
- incertidumbre predictiva, aleatorica y epistemica;
- calibracion, contrafactuales y resumen por tipo de error.
- Faithfulness Correlation formal para cumplir la métrica cuantitativa de fidelidad solicitada.


## Estructura

Los archivos usados por este notebook estan en:

```text
notebook/
├── xai_resultados_avanzados.ipynb
├── generar_notebook_resultados.py
├── README.md
└── resultados/
    ├── README.md
    ├── figuras/
    └── tablas/
```

Si se quiere reconstruir la carpeta `resultados/`, ejecutar:

```bash
python notebook/generar_notebook_resultados.py
```


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'resultados').exists() and (NOTEBOOK_DIR / 'notebook' / 'resultados').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebook'

RESULTS_DIR = NOTEBOOK_DIR / 'resultados'
TABLES_DIR = RESULTS_DIR / 'tablas'
FIGURES_DIR = RESULTS_DIR / 'figuras'

print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('tablas =', len(list(TABLES_DIR.glob('*.csv'))))
print('figuras =', len(list(FIGURES_DIR.glob('*.png'))))


## Resumen ejecutivo

1. **Occlusion** sigue siendo el metodo XAI mas fiel por intervencion.
2. **Grad-CAM clasico** es inestable, especialmente en TESS; por eso se usa solo como diagnostico.
3. **Grad-CAM multiescala** mejora la localizacion temporal y se vuelve mas defendible.
4. El conteo de focos muestra que los errores no necesariamente tienen mas focos; lo mas claro es que los `TN` tienen atencion mas dispersa y menor importancia central.
5. En incertidumbre, **MC Dropout muestra epistemica baja** en promedio. La incertidumbre de los errores aparece principalmente como mayor entropia predictiva/aleatorica, no como gran epistemica.


## MaxLike y MC Dropout

**MaxLike** corresponde a la inferencia deterministica del modelo con `model.eval()` y dropout apagado. Es la probabilidad puntual del modelo.

**MC Dropout** mantiene dropout activo durante inferencia y realiza muchas predicciones. A partir de esas muestras se calcula:

- `predictive_entropy = H(E[p])`: incertidumbre predictiva total;
- `aleatoric_uncertainty = E[H(p_t)]`: incertidumbre esperada de los datos;
- `epistemic_uncertainty = H(E[p]) - E[H(p_t)]`: incertidumbre del modelo.

En estos resultados la epistemica es baja, lo que sugiere que el dropout no esta generando modelos muy distintos entre si. Para trabajo futuro se podria comparar con Deep Ensembles.


### Incertidumbre promedio por dataset

| dataset | n | maxlike_confidence_mean | maxlike_entropy_mean | mc_dropout_std_mean | predictive_entropy_mean | aleatoric_uncertainty_mean | epistemic_uncertainty_mean |
| --- | --- | --- | --- | --- | --- | --- | --- |
| kepler | 1574 | 0.9615 | 0.0954 | 0.0041 | 0.0963 | 0.096 | 0.0003 |
| tess | 1655 | 0.9658 | 0.0755 | 0.0034 | 0.0761 | 0.0759 | 0.0003 |


### Incertidumbre por tipo de resultado

| dataset | outcome | n | maxlike_confidence_mean | maxlike_entropy_mean | mc_dropout_std_mean | predictive_entropy_mean | aleatoric_uncertainty_mean | epistemic_uncertainty_mean |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| kepler | FN | 17 | 0.7361 | 0.4869 | 0.0253 | 0.4866 | 0.4846 | 0.002 |
| kepler | FP | 35 | 0.7922 | 0.438 | 0.0177 | 0.442 | 0.4408 | 0.0012 |
| kepler | TN | 1179 | 0.9861 | 0.0353 | 0.0017 | 0.0355 | 0.0354 | 0.0001 |
| kepler | TP | 343 | 0.9058 | 0.2475 | 0.01 | 0.2505 | 0.2498 | 0.0008 |
| tess | FN | 20 | 0.6137 | 0.5423 | 0.0239 | 0.5437 | 0.5421 | 0.0016 |
| tess | FP | 28 | 0.889 | 0.3247 | 0.0127 | 0.3256 | 0.3247 | 0.0009 |
| tess | TN | 1348 | 0.9705 | 0.0619 | 0.0029 | 0.0626 | 0.0623 | 0.0002 |
| tess | TP | 259 | 0.9767 | 0.0832 | 0.0031 | 0.0838 | 0.0835 | 0.0003 |


### Figuras MaxLike vs MC Dropout

![Kepler MaxLike vs MC](resultados/figuras/kepler_maxlike_vs_mc_dropout.png)

![TESS MaxLike vs MC](resultados/figuras/tess_maxlike_vs_mc_dropout.png)


### Descomposicion de incertidumbre por outcome

![Kepler uncertainty](resultados/figuras/kepler_uncertainty_decomposition_by_outcome.png)

![TESS uncertainty](resultados/figuras/tess_uncertainty_decomposition_by_outcome.png)


## Focos de atencion

Se calculo `Saliency` target-aware para todo el test set. Luego se contaron componentes conectados de alta relevancia en la vista global y local. Esto produce una metrica exploratoria:

- `total_focus_count`: numero total de focos global+local;
- `attention_position_variance`: dispersion temporal ponderada por relevancia;
- `attention_entropy`: entropia espacial del mapa;
- `central_importance_ratio`: fraccion de relevancia cerca del transito.

Esto no reemplaza metricas de rendimiento, pero sirve para discutir si los errores muestran patrones de atencion diferentes.


### Resumen de focos por outcome

| dataset | outcome | n | total_focus_count_mean | total_focus_count_std | total_focus_count_var | global_focus_count_mean | local_focus_count_mean | attention_position_variance_mean | attention_position_variance_std | attention_entropy_mean | central_importance_ratio_mean | mc_std_mean | target_confidence_mean |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| kepler | FN | 17 | 5.4118 | 1.5835 | 2.5074 | 2.4118 | 3.0 | 0.0532 | 0.0222 | 0.8113 | 0.4826 | 0.0253 | 0.7361 |
| kepler | FP | 35 | 5.7143 | 2.0803 | 4.3277 | 2.6286 | 3.0857 | 0.0549 | 0.0166 | 0.8345 | 0.4076 | 0.0177 | 0.7922 |
| kepler | TN | 1179 | 5.6819 | 1.7884 | 3.1984 | 2.5674 | 3.1145 | 0.0752 | 0.0204 | 0.857 | 0.25 | 0.0017 | 0.9861 |
| kepler | TP | 343 | 6.2303 | 1.8328 | 3.3591 | 2.4869 | 3.7434 | 0.0539 | 0.0162 | 0.826 | 0.4103 | 0.01 | 0.9058 |
| tess | FN | 20 | 8.15 | 2.0072 | 4.0289 | 3.95 | 4.2 | 0.0483 | 0.0112 | 0.9005 | 0.3817 | 0.0238 | 0.6137 |
| tess | FP | 28 | 7.75 | 1.7132 | 2.9352 | 3.7857 | 3.9643 | 0.0467 | 0.0091 | 0.8989 | 0.3518 | 0.0127 | 0.889 |
| tess | TN | 1348 | 8.181 | 2.0628 | 4.2553 | 3.7062 | 4.4748 | 0.0639 | 0.0122 | 0.9202 | 0.2557 | 0.0029 | 0.9705 |
| tess | TP | 259 | 8.5521 | 1.8385 | 3.38 | 3.8649 | 4.6873 | 0.0498 | 0.0093 | 0.9021 | 0.3518 | 0.0031 | 0.9767 |


### Matriz de confusion normalizada con focos de atencion

Cada fila esta normalizada por la etiqueta real, por lo que los porcentajes de cada fila suman 100%. El color representa esa proporcion normalizada. El texto de cada celda mantiene `n`, media/desviacion de focos y dispersion temporal de la atencion.

![Kepler focus confusion](resultados/figuras/kepler_attention_focus_confusion_matrix.png)

![TESS focus confusion](resultados/figuras/tess_attention_focus_confusion_matrix.png)


### Distribucion por outcome

![Kepler focus boxplot](resultados/figuras/kepler_attention_focus_outcome_boxplot.png)

![TESS focus boxplot](resultados/figuras/tess_attention_focus_outcome_boxplot.png)


## Mapas XAI agregados

Estas figuras comparan Saliency, SmoothGrad, Integrated Gradients, Occlusion, Grad-CAM, Grad-CAM multiescala y Consenso.

![Kepler aggregate](resultados/figuras/kepler_aggregate_importance_selected_examples.png)

![TESS aggregate](resultados/figuras/tess_aggregate_importance_selected_examples.png)


## Fidelidad por borrado

La prueba de borrado elimina progresivamente los puntos mas relevantes segun cada metodo. Si el mapa es fiel, la confianza hacia la clase explicada deberia caer rapidamente.

![Kepler deletion](resultados/figuras/kepler_faithfulness_deletion_curves.png)

![TESS deletion](resultados/figuras/tess_faithfulness_deletion_curves.png)


## Faithfulness Correlation formal

Para cumplir la evaluación cuantitativa de fidelidad se calculó una **Faithfulness Correlation** estilo Bhatt et al. (2021). La idea es simple: si un método XAI marca ciertos puntos temporales como importantes, al perturbar esos puntos la confianza del modelo hacia la clase explicada debería cambiar más que al perturbar puntos poco importantes.

Implementación usada en esta entrega: para cada ejemplo seleccionado se hicieron 40 perturbaciones aleatorias, apagando 10% de la vista global y 10% de la vista local. Luego se correlacionó la suma de atribución dentro de los puntos perturbados con la caída de confianza del modelo hacia la clase explicada. Se reportan Pearson y Spearman.

Un valor positivo alto indica mayor fidelidad. Un valor cercano a cero indica asociación débil entre mapa XAI y comportamiento del modelo. Un valor negativo indica que el método puede estar resaltando regiones que no explican bien la predicción bajo esta perturbación.


### Resultados de Faithfulness Correlation

| dataset | method | valid_n | pearson_mean | spearman_mean | mean_confidence_drop |
| --- | --- | --- | --- | --- | --- |
| kepler | integrated_gradients | 5 | 0.2115 | 0.2462 | 0.0300 |
| kepler | occlusion | 5 | 0.2109 | 0.1594 | 0.0300 |
| kepler | consensus | 5 | 0.1961 | 0.1536 | 0.0300 |
| kepler | saliency | 5 | 0.1434 | 0.1699 | 0.0300 |
| kepler | gradcam_multiscale | 5 | 0.0496 | 0.0565 | 0.0300 |
| tess | occlusion | 6 | 0.4957 | 0.5138 | -0.0667 |
| tess | consensus | 6 | 0.3654 | 0.4209 | -0.0667 |
| tess | gradcam_multiscale | 6 | 0.3154 | 0.3736 | -0.0667 |
| tess | saliency | 6 | 0.2049 | 0.2050 | -0.0667 |
| tess | integrated_gradients | 6 | 0.1806 | 0.2574 | -0.0667 |

En **Kepler**, el mejor valor promedio fue `integrated_gradients` con Pearson `0.2115` y Spearman `0.2462`. Esto indica fidelidad positiva pero baja/moderada: las regiones importantes tienden a afectar la confianza, aunque no de forma fuerte ni perfectamente consistente.

En **TESS**, el mejor valor promedio fue `occlusion` con Pearson `0.4957` y Spearman `0.5138`. Este resultado es más defendible que Kepler para ese método, especialmente para `occlusion`, porque la correlación supera claramente cero.

La columna `mean_confidence_drop` debe interpretarse con cuidado. En TESS aparece negativa en promedio, lo que significa que varias perturbaciones aleatorias aumentaron la confianza hacia la clase explicada en vez de reducirla. Eso no invalida la correlación, pero sí muestra que el modelo puede ser sensible a perturbaciones fuera de distribución y que el signo de la caída no siempre es estable.


### Figuras de Faithfulness Correlation

![Kepler faithfulness correlation](resultados/figuras/kepler_faithfulness_correlation_by_method.png)

![TESS faithfulness correlation](resultados/figuras/tess_faithfulness_correlation_by_method.png)


### Limitaciones de la métrica

Esta métrica se debe reportar como evidencia complementaria, no como validación absoluta del XAI. En este problema hay tres limitaciones importantes. Primero, apagar puntos temporales con valor cero puede generar curvas de luz artificiales que no siguen exactamente la distribución real de Kepler o TESS. Segundo, el modelo es una CNN no lineal con pooling, bloques SE y capas densas residuales; por lo tanto, la importancia de un punto depende de interacciones con otros puntos y no se puede aislar perfectamente con perturbaciones independientes. Tercero, la evaluación se calculó sobre ejemplos representativos seleccionados, no sobre todo el test set, porque recalcular mapas y perturbaciones para todo el conjunto es costoso.

Esto se alinea con la advertencia metodológica reciente: las métricas de fidelidad suelen ser más confiables para modelos lineales que para redes neuronales profundas no lineales, como discuten Miró-Nicolau et al. (2025). Por eso, en el informe conviene presentar Faithfulness Correlation junto con la prueba de borrado, Occlusion, Integrated Gradients y el análisis visual, no como única prueba.


## Calibracion

La calibracion se hizo con temperature scaling usando validacion. No cambia pesos del modelo. En estos resultados no mejora claramente el ECE, por lo que se reporta como auditoria, no como mejora principal.

| dataset | split | calibration | temperature | n | ece | mce | brier | nll |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| kepler | test | raw | 1.0 | 1574 | 0.0111 | 0.2171 | 0.0283 | 0.1025 |
| kepler | test | temperature_scaled | 1.3465 | 1574 | 0.0206 | 0.2384 | 0.0294 | 0.1055 |
| tess | test | raw | 1.0 | 1655 | 0.028 | 0.5558 | 0.0284 | 0.0943 |
| tess | test | temperature_scaled | 1.2809 | 1655 | 0.031 | 0.445 | 0.0278 | 0.0935 |


![Kepler reliability](resultados/figuras/kepler_reliability_calibration.png)

![TESS reliability](resultados/figuras/tess_reliability_calibration.png)


## Contrafactuales simples

Se modifico la ventana central del transito: aplanar, profundizar o reducir la profundidad. Esto mide si la probabilidad cambia de forma coherente cuando se altera la morfologia transitiva.

![Kepler counterfactual](resultados/figuras/kepler_counterfactual_effects.png)

![TESS counterfactual](resultados/figuras/tess_counterfactual_effects.png)


## Ejemplos individuales seleccionados

Estas figuras muestran casos representativos: positivo correcto, negativo correcto, cerca del umbral, falso positivo, falso negativo y mayor incertidumbre MC Dropout.


In [ ]:
for path in sorted(FIGURES_DIR.glob('*_xai_maps.png')):
    print(path.name)
    display(Image(filename=str(path)))


## Cargar tablas completas

Las tablas completas estan disponibles en `resultados/tablas/`. Esta celda permite inspeccionarlas sin salir del notebook.


In [ ]:
tablas = {p.stem: pd.read_csv(p) for p in sorted(TABLES_DIR.glob('*.csv'))}
print('
'.join(f'{k}: {v.shape}' for k, v in tablas.items()))


In [ ]:
# Ejemplos utiles para explorar manualmente:
# tablas['xai_attention_focus_by_sample'].head()
# tablas['uncertainty_maxlike_mc_dropout_by_sample'].query("outcome in ['FP', 'FN']").sort_values('epistemic_uncertainty', ascending=False).head()
# tablas['xai_faithfulness_metrics'].head()


## Métricas XAI adicionales de alto nivel

Para dejar el análisis más robusto se agregaron métricas cuantitativas derivadas:

- `Faithfulness AUC`: área normalizada bajo la curva de caída de confianza al borrar puntos importantes.
- `Monotonicidad de borrado`: qué tan consistentemente cae la confianza al borrar más puntos.
- `Agreement entre métodos`: solapamiento top-10 entre mapas de relevancia.
- `Centralidad`: fracción de importancia cerca del tránsito por método y vista.
- `Attention quality/risk score`: resumen normalizado de centralidad, dispersión, entropía, cantidad de focos, MC std y baja confianza.
- `Error detection AUC`: qué tanto una métrica XAI/incertidumbre separa errores (`FP/FN`) de aciertos (`TP/TN`).


### Fidelidad agregada por AUC de borrado

Mientras más alto el AUC, más cae la confianza cuando se borran las zonas que el método considera importantes.

| dataset | method | faithfulness_auc_mean | drop_at_10_mean | max_drop_mean | negative_drop_fraction_mean | monotonicity_fraction_mean |
| --- | --- | --- | --- | --- | --- | --- |
| kepler | consensus | 0.4588 | 0.5512 | 0.617 | 0.0833 | 0.6333 |
| kepler | occlusion | 0.4587 | 0.5088 | 0.6213 | 0.0556 | 0.8 |
| kepler | gradcam_multiscale | 0.3814 | 0.447 | 0.599 | 0.1111 | 0.7 |
| kepler | integrated_gradients | 0.3282 | 0.3634 | 0.4488 | 0.1389 | 0.6333 |
| kepler | saliency | 0.2808 | 0.1421 | 0.5878 | 0.1389 | 0.7 |
| kepler | smoothgrad | 0.2592 | 0.1231 | 0.5715 | 0.1389 | 0.6 |
| kepler | gradcam | 0.106 | 0.0966 | 0.3172 | 0.1944 | 0.7667 |
| tess | occlusion | 0.3863 | 0.4347 | 0.502 | 0.0 | 0.6667 |
| tess | gradcam_multiscale | 0.2133 | 0.2392 | 0.3635 | 0.2778 | 0.6 |
| tess | consensus | 0.1421 | 0.1895 | 0.3586 | 0.2778 | 0.5333 |
| tess | integrated_gradients | 0.1118 | 0.1234 | 0.3515 | 0.2778 | 0.6333 |
| tess | smoothgrad | 0.0906 | 0.0715 | 0.3351 | 0.2778 | 0.7 |
| tess | saliency | 0.0869 | 0.0494 | 0.3458 | 0.2778 | 0.6333 |
| tess | gradcam | -0.1469 | -0.1453 | 0.0369 | 0.5 | 0.4667 |


![Kepler faithfulness AUC](resultados/figuras/kepler_faithfulness_auc_by_method.png)

![TESS faithfulness AUC](resultados/figuras/tess_faithfulness_auc_by_method.png)


### Centralidad por método

Esta tabla muestra qué porcentaje de la importancia cae cerca del tránsito. Sirve para ver qué métodos están mirando la región física esperada.

| dataset | view | saliency_central_importance_ratio | smoothgrad_central_importance_ratio | integrated_gradients_central_importance_ratio | gradcam_central_importance_ratio | gradcam_multiscale_central_importance_ratio | occlusion_central_importance_ratio | consensus_central_importance_ratio |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| kepler | global | 0.3425 | 0.3539 | 0.2505 | 0.2434 | 0.3924 | 0.2492 | 0.2945 |
| kepler | local | 0.5326 | 0.5468 | 0.7169 | 0.433 | 0.7177 | 0.4943 | 0.6208 |
| tess | global | 0.2975 | 0.2953 | 0.5972 | 0.1098 | 0.2603 | 0.5953 | 0.3418 |
| tess | local | 0.3191 | 0.3272 | 0.6088 | 0.2075 | 0.39 | 0.4074 | 0.4092 |


![Kepler centrality](resultados/figuras/kepler_xai_method_centrality.png)

![TESS centrality](resultados/figuras/tess_xai_method_centrality.png)


### Quality score y risk score de atención

El `attention_quality_score` combina mayor centralidad, menor dispersión, menor entropía y menor cantidad de focos. El `attention_risk_score` combina baja centralidad, alta dispersión, alta entropía, muchos focos, mayor MC std y menor confianza.

| dataset | outcome | n | attention_quality_mean | attention_quality_std | attention_risk_mean | attention_risk_std | total_focus_count_mean | attention_position_variance_mean | attention_entropy_mean | central_importance_ratio_mean |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| kepler | FN | 17 | 0.6211 | 0.1194 | 0.4018 | 0.0971 | 5.4118 | 0.0532 | 0.8113 | 0.4826 |
| kepler | FP | 35 | 0.5641 | 0.1291 | 0.4028 | 0.0995 | 5.7143 | 0.0549 | 0.8345 | 0.4076 |
| kepler | TN | 1179 | 0.456 | 0.1197 | 0.3713 | 0.0773 | 5.6819 | 0.0752 | 0.857 | 0.25 |
| kepler | TP | 343 | 0.5607 | 0.1084 | 0.3484 | 0.0873 | 6.2303 | 0.0539 | 0.826 | 0.4103 |
| tess | FN | 20 | 0.5514 | 0.1309 | 0.4467 | 0.1096 | 8.15 | 0.0483 | 0.9005 | 0.3817 |
| tess | FP | 28 | 0.5534 | 0.1129 | 0.3555 | 0.0773 | 7.75 | 0.0467 | 0.8989 | 0.3518 |
| tess | TN | 1348 | 0.4008 | 0.1113 | 0.4135 | 0.0723 | 8.181 | 0.0639 | 0.9202 | 0.2557 |
| tess | TP | 259 | 0.5213 | 0.1214 | 0.3325 | 0.0855 | 8.5521 | 0.0498 | 0.9021 | 0.3518 |


![Kepler quality risk](resultados/figuras/kepler_attention_quality_risk_scores.png)

![TESS quality risk](resultados/figuras/tess_attention_quality_risk_scores.png)


### Métricas XAI para detectar errores

Esta tabla evalúa si cada métrica separa errores (`FP/FN`) de aciertos (`TP/TN`). `roc_auc_best_direction` se reporta tomando la mejor dirección posible, por lo que valores cercanos a 1 indican mayor poder separador. Esto es útil como future work: usar estas métricas como detector de predicciones poco confiables.

| dataset | metric | n_error | n_correct | mean_error | mean_correct | cohen_d_error_minus_correct | roc_auc_best_direction | risk_direction |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| kepler | aleatoric_uncertainty | 52 | 1522 | 0.4551 | 0.0837 | 2.2378 | 0.913 | high=risk |
| kepler | target_confidence | 52 | 1522 | 0.7739 | 0.968 | -2.2313 | 0.9128 | low=risk |
| kepler | maxlike_entropy | 52 | 1522 | 0.454 | 0.0831 | 2.236 | 0.9128 | high=risk |
| kepler | mc_std | 52 | 1522 | 0.0202 | 0.0035 | 2.0058 | 0.9084 | high=risk |
| kepler | mc_dropout_std_probability | 52 | 1522 | 0.0202 | 0.0035 | 2.0058 | 0.9084 | high=risk |
| kepler | epistemic_uncertainty | 52 | 1522 | 0.0015 | 0.0003 | 1.4315 | 0.8991 | high=risk |
| kepler | mean_central_importance_ratio | 52 | 1522 | 0.4321 | 0.2861 | 0.9672 | 0.7517 | high=risk |
| kepler | attention_quality_score | 52 | 1522 | 0.5827 | 0.4796 | 0.8236 | 0.7223 | high=risk |
| kepler | mean_attention_position_variance | 52 | 1522 | 0.0543 | 0.0704 | -0.7546 | 0.7113 | low=risk |
| kepler | mean_attention_entropy | 52 | 1522 | 0.8269 | 0.85 | -0.5642 | 0.67 | low=risk |
| kepler | attention_risk_score | 52 | 1522 | 0.4025 | 0.3662 | 0.4493 | 0.6006 | high=risk |
| kepler | total_focus_count | 52 | 1522 | 5.6154 | 5.8055 | -0.1047 | 0.5343 | low=risk |
| tess | target_confidence | 48 | 1607 | 0.7743 | 0.9715 | -1.9471 | 0.9287 | low=risk |
| tess | aleatoric_uncertainty | 48 | 1607 | 0.4153 | 0.0657 | 2.37 | 0.9276 | high=risk |
| tess | maxlike_entropy | 48 | 1607 | 0.4153 | 0.0653 | 2.373 | 0.9275 | high=risk |
| tess | mc_std | 48 | 1607 | 0.0174 | 0.0029 | 1.977 | 0.9238 | high=risk |
| tess | mc_dropout_std_probability | 48 | 1607 | 0.0174 | 0.0029 | 1.9748 | 0.9237 | high=risk |
| tess | epistemic_uncertainty | 48 | 1607 | 0.0012 | 0.0002 | 1.5406 | 0.9152 | high=risk |
| tess | mean_attention_position_variance | 48 | 1607 | 0.0474 | 0.0616 | -1.1138 | 0.8189 | low=risk |
| tess | attention_quality_score | 48 | 1607 | 0.5526 | 0.4202 | 1.0915 | 0.7819 | high=risk |
| tess | mean_central_importance_ratio | 48 | 1607 | 0.3643 | 0.2712 | 1.0679 | 0.7729 | high=risk |
| tess | mean_attention_entropy | 48 | 1607 | 0.8996 | 0.9173 | -0.8746 | 0.7279 | low=risk |
| tess | attention_risk_score | 48 | 1607 | 0.3935 | 0.4004 | -0.0857 | 0.5543 | low=risk |
| tess | total_focus_count | 48 | 1607 | 7.9167 | 8.2408 | -0.1599 | 0.5492 | low=risk |


![Kepler error detection](resultados/figuras/kepler_xai_error_detection_auc.png)

![TESS error detection](resultados/figuras/tess_xai_error_detection_auc.png)


In [ ]:
# Tablas nuevas disponibles:
extra_tables = [
    'xai_faithfulness_auc_by_sample',
    'xai_faithfulness_auc_summary',
    'xai_faithfulness_correlation_by_sample',
    'xai_faithfulness_correlation_summary',
    'xai_attention_quality_by_sample',
    'xai_attention_quality_summary',
    'xai_error_detection_metrics',
    'xai_method_agreement_summary',
    'xai_method_centrality_summary',
]
for name in extra_tables:
    path = TABLES_DIR / f'{name}.csv'
    df = pd.read_csv(path)
    print(f'{name}: {df.shape}')


## Figuras académicas bilingües listas para paper

Se creó un paquete bilingüe de figuras con nombres académicos y estructura separada para paper/informe. Las figuras principales están regeneradas con rótulos formales; el material suplementario contiene todas las figuras obtenidas durante el análisis, copiadas con nombres académicos en español e inglés.

```text
resultados/figuras_academicas/
├── indice_figuras_academicas.csv
├── espanol/
│   ├── figuras_principales/
│   └── material_suplementario/
└── english/
    ├── main_figures/
    └── supplementary_figures/
```

El archivo `indice_figuras_academicas.csv` permite rastrear cada figura desde su nombre original hasta su versión académica.

### Figuras principales en español

![Kepler matriz ES](resultados/figuras_academicas/espanol/figuras_principales/kepler_figura_principal_matriz_confusion_normalizada_atencion_xai.png)

![TESS matriz ES](resultados/figuras_academicas/espanol/figuras_principales/tess_figura_principal_matriz_confusion_normalizada_atencion_xai.png)

![Kepler riesgo ES](resultados/figuras_academicas/espanol/figuras_principales/kepler_figura_principal_violin_score_riesgo_atencion_xai.png)

![TESS riesgo ES](resultados/figuras_academicas/espanol/figuras_principales/tess_figura_principal_violin_score_riesgo_atencion_xai.png)

![Kepler focos-varianza ES](resultados/figuras_academicas/espanol/figuras_principales/kepler_figura_principal_violin_focos_y_varianza_atencion.png)

![TESS focos-varianza ES](resultados/figuras_academicas/espanol/figuras_principales/tess_figura_principal_violin_focos_y_varianza_atencion.png)

### Main figures in English

![Kepler matrix EN](resultados/figuras_academicas/english/main_figures/kepler_main_figure_row_normalized_confusion_matrix_xai_attention.png)

![TESS matrix EN](resultados/figuras_academicas/english/main_figures/tess_main_figure_row_normalized_confusion_matrix_xai_attention.png)

![Kepler risk EN](resultados/figuras_academicas/english/main_figures/kepler_main_figure_xai_attention_risk_score_violin.png)

![TESS risk EN](resultados/figuras_academicas/english/main_figures/tess_main_figure_xai_attention_risk_score_violin.png)

![Kepler focus variance EN](resultados/figuras_academicas/english/main_figures/kepler_main_figure_attention_focus_count_and_variance_violin.png)

![TESS focus variance EN](resultados/figuras_academicas/english/main_figures/tess_main_figure_attention_focus_count_and_variance_violin.png)


## Uso de XAI para mejorar el modelo

Las explicaciones si pueden ayudar a mejorar el desempeno, pero deben usarse como senales auxiliares y no como etiquetas verdaderas. En la practica sirven para detectar curvas mal centradas, ejemplos con ruido, posibles etiquetas dudosas, falsos positivos que miran regiones no fisicas y falsos negativos con atencion dispersa.

La propuesta mas defendible como trabajo futuro es usar las metricas XAI como un detector de riesgo: combinar cantidad de focos, dispersion, entropia de atencion, centralidad, fidelidad, MC Dropout y confianza para decidir si una prediccion debe aceptarse directamente, cambiar de umbral o pasar a revision. Otra linea posible es usar los mapas para mejorar el preprocesamiento o aplicar regularizacion explicable, pero eso debe validarse siempre en test porque las explicaciones de una CNN no lineal no son prueba causal absoluta.


## Interpretacion para informe

La interpretacion mas defendible es:

- los mapas basados en intervencion (`Occlusion`) y acumulacion de gradientes (`Integrated Gradients`) son los mas estables;
- Grad-CAM clasico puede fallar en series temporales con pooling fuerte;
- Grad-CAM multiescala corrige parcialmente ese problema;
- los errores no se explican solo por mayor cantidad de focos, pero si muestran diferencias en confianza, entropia e importancia central;
- el conteo de focos puede proponerse como trabajo futuro para construir una alerta de predicciones poco confiables.
